# Advanced document indexing

# Splitting and ingesting HTML content

## Splitting and ingesting the content of a single URL (on Cornwall)

### Preparing the Chroma DB collections

In [1]:
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import getpass

GOOGLE_API_KEY = getpass.getpass('Enter your GOOGLE_API_KEY')

In [2]:
corwnall_granular_collection = Chroma( #A
    collection_name="cornwall_granular",
    embedding_function=GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=GOOGLE_API_KEY),
)

corwnall_granular_collection.reset_collection() #B
# A Create a Chorma DB collection
# B Reset the collection in case it already exists 

In [3]:
corwnall_coarse_collection = Chroma( #A 
    collection_name="cornwall_coarse",
    embedding_function=GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=GOOGLE_API_KEY),
)

corwnall_coarse_collection.reset_collection() #B
# A Create a Chorma DB collection
# B Reset the collection in case it already exists 

### Loading the HTML content with the AsyncHtmlLoader 

In [3]:
from langchain_community.document_loaders import AsyncHtmlLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [4]:
destination_url = "https://en.wikivoyage.org/wiki/Cornwall"

In [28]:
header_template = {"User-Agent": " MiScriptEducativo/1.0 (ricardo.ares1989@gmail.com)"}
html_loader = AsyncHtmlLoader(
    destination_url,
    header_template=header_template

)

In [6]:
docs = html_loader.load()

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.86it/s]


In [7]:
docs

[Document(metadata={'source': 'https://en.wikivoyage.org/wiki/Cornwall', 'title': 'Cornwall – Travel guide at Wikivoyage', 'language': 'en'}, page_content='<!DOCTYPE html>\n<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-available skin-theme-clientpref-thumb-standard" lang="en" dir="ltr">\n<head>\n<meta charset="UTF-8">\n<title>Cornwall – Travel guide at Wikivoyage</title>\n<script>(function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vec

M### Splitting into granular chunks with the HTMLSectionSplitter

In [8]:
from langchain_text_splitters import HTMLSectionSplitter

In [9]:
headers_to_split_on = [("h1", "Header 1"), ("h2", "Header 2")]
html_section_splitter = HTMLSectionSplitter(
    headers_to_split_on=headers_to_split_on)

In [10]:
def split_docs_into_granular_chunks(docs):
    all_chunks = []
    for doc in docs:
        html_string = doc.page_content #A
        temp_chunks = html_section_splitter.split_text(
            html_string) #B
        all_chunks.extend(temp_chunks) 

    return all_chunks

#A Extract the HTML text from the document
#B Each chunk is a H1 or H2 HTML section

In [11]:
granular_chunks = split_docs_into_granular_chunks(docs)

In [12]:
granular_chunks

[Document(metadata={'Header 1': '#TITLE#'}, page_content='Jump to content \n \n \n \n \n \n \n \n Main menu \n \n \n \n \n \n Main menu \n move to sidebar \n hide \n \n \n \n\t\tNavigation\n\t \n \n \n Main page \n Travel destinations \n Star articles \n What\'s nearby? \n Travel forum \n Arrivals lounge \n Random page \n \n \n \n \n \n\t\tGet involved\n\t \n \n \n Travellers\' pub \n Recent changes \n Community portal \n Maintenance panel \n Policies \n Help \n Interlingual lounge \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n Search \n \n \n \n \n \n \n \n \n \n \n \n Search \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n Appearance \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n Donate \n \n \n Create account \n \n \n Log in \n \n \n \n \n \n \n \n \n Personal tools \n \n \n \n \n \n Donate \n   Create account \n   Log in \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n (function(){var node=document.getElementById("mw-dismissablenotice-anonplace");if

#### Ingesting granular chunks

In [13]:
corwnall_granular_collection.add_documents(documents=granular_chunks)

['f89ebc25-1e14-4f49-9461-e8185ca6ec30',
 '6ce38f4c-9a57-446a-bde6-c9f6ce545f22',
 '63837644-441a-4991-bb1f-9f808254568b',
 '5bd16df4-54a1-47e1-b1b0-0b14c4588325',
 '6eb93b2e-2477-4d42-9730-74e839a90f01',
 '4f1be437-789c-4303-9880-f4b304837b63',
 '82f2983d-c859-4f81-b138-b30ff5302199',
 '9b4cacb1-7c28-40cb-a185-ef5762238286',
 '38c381a1-2660-4f41-a7e6-141709094bbf',
 '12b3c6f5-7cb1-494b-9d3a-1058e3da9300',
 'c213b2a0-a620-4398-93b9-26558624cf31',
 '4e8a024a-4010-4c90-9c3e-8ee3074060dc',
 'de73f54a-9b65-49a2-bfd9-6c4ea5e247c6',
 '0daec602-41ff-4a04-9a53-91122259ce86',
 '5dffe65c-b9a4-4304-a65a-ff346549239a',
 '4eef6ef8-8bca-4145-b085-dc5d4a0cbae0',
 '2dd08a98-e684-4bac-adf5-022f27a7a1ec',
 '43970189-cd34-40ff-842d-6e0410f9e4de',
 '2c9aeecf-0994-4938-90e6-3d00b26403b5']

#### Searching granular chunks

In [14]:
results = corwnall_granular_collection.similarity_search(
    query="Events or festivals in Cornwall",k=3)
for doc in results:
    print(doc)

page_content='Festivals 
 [ edit ] 
 
 These festivals tend to not be public holidays and not all are celebrated fully across the county. 
   
 AberFest .   A Celtic cultural festival celebrating “All things” Cornish and Breton that takes place biennially (every two years) in Cornwall at Easter. The AberFest Festival alternates with the Breizh – Kernow Festival that is held in Brandivy and Bignan (in Breizh/Bretagne – France) on the alternate years.       ( updated Jun 2023 ) 
 Golowan , sometimes also  Goluan  or  Gol-Jowan  is the Cornish word for the Midsummer celebrations, most popular in the Penwith area and in particular  Penzance  and  Newlyn . The celebrations are conducted from the 23rd of June (St John's Eve) to the 28th of June (St Peter's Eve) each year, St Peter's Eve being the more popular in Cornish fishing communities. The celebrations are centred around the lighting of bonfires and fireworks and the performance of associated rituals. Some towns have a street-parade dur

### Splitting into coarse chunks with the RecursiveCharacterTextSplitter 

In [15]:
from langchain_community.document_transformers import Html2TextTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [16]:
html2text_transformer = Html2TextTransformer()

In [17]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000, chunk_overlap=300
)

In [18]:
def split_docs_into_coarse_chunks(docs):
    text_docs = html2text_transformer.transform_documents(
        docs) #A 
    coarse_chunks = text_splitter.split_documents(
        text_docs)

    return coarse_chunks
#A transform HTML docs into clean text docs

In [20]:
coarse_chunks = split_docs_into_coarse_chunks(docs)

#### Ingesting coarse chunks

In [22]:

corwnall_coarse_collection = Chroma( #A
    collection_name="cornwall_coarse",
    embedding_function=GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=GOOGLE_API_KEY),
)

corwnall_coarse_collection.reset_collection()
corwnall_coarse_collection.add_documents(documents=coarse_chunks)

['9abc9c98-d227-4cea-9185-8eed4be5be56',
 'eb4f46f2-4d7d-4c56-9ca3-978ffdd36c21',
 'f8024e48-c600-449e-9b03-27c4e3a16dba',
 '698bdc94-2424-4271-8b74-fde56b478766',
 'c94ae5ae-fab4-4126-a9d7-c21bb923af77',
 '459a8420-3779-4057-8882-b9185bcdadf7',
 'cba4aa18-1a49-4b7d-b965-c21c25ba98c0',
 '17b636dd-faec-4438-9e2e-288531d8ae88',
 'a60184bd-5c35-4a33-9d6c-63829cb4a128',
 '95e64180-befc-431d-b6d4-71daebffcfb0',
 '2c70c353-bdae-46c4-a975-8fa651916e27',
 'fd33e357-ab4f-4cfc-8118-68609c0febac',
 'e1a15b3f-9e7b-4150-a9ea-77197827f39e',
 '5d5fa9b8-218d-41e5-be0b-05ac6e0ff122',
 '12f69474-c19c-45a1-8150-8c1a9684b8b9']

#### Searching coarse chunks

In [23]:
results = corwnall_coarse_collection.similarity_search(
    query="Events or festivals in Cornwall",k=3)
for doc in results:
    print(doc)

page_content='### Spirits

[edit]

    _See also:Liquor_

Gin and rum are also produced in Cornwall. A popular brand of Cornish rum is
Dead Man's Fingers which has multiple flavours and is bottled in St. Ives.

## Festivals

[edit]

These festivals tend to not be public holidays and not all are celebrated
fully across the county.

AberFest. A Celtic cultural festival celebrating “All things” Cornish and
Breton that takes place biennially (every two years) in Cornwall at Easter.
The AberFest Festival alternates with the Breizh – Kernow Festival that is
held in Brandivy and Bignan (in Breizh/Bretagne – France) on the alternate
years.  (updated Jun 2023)

**Golowan** , sometimes also _Goluan_ or _Gol-Jowan_ is the Cornish word for
the Midsummer celebrations, most popular in the Penwith area and in particular
Penzance and Newlyn. The celebrations are conducted from the 23rd of June (St
John's Eve) to the 28th of June (St Peter's Eve) each year, St Peter's Eve
being the more popular in Corn

## Splitting and ingesting the content of various URLs (across UK destinations)

### Preparing the Chroma DB collections

In [24]:
uk_granular_collection = Chroma( #A
    collection_name="uk_granular",
    embedding_function=GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=GOOGLE_API_KEY),
)

uk_granular_collection.reset_collection() #B

In [25]:
uk_coarse_collection = Chroma( #A
    collection_name="uk_coarse",
    embedding_function=GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=GOOGLE_API_KEY),
)

uk_coarse_collection.reset_collection() #B

### Splitting and ingesting HTML content with the HTMLSectionSplitter 

In [26]:
# Reduce this list if you want to save on processing fees
uk_destinations = [
    "Cornwall", "North_Cornwall", "South_Cornwall", "West_Cornwall", 
    "Tintagel", "Bodmin", "Wadebridge", "Penzance", "Newquay",
    "St_Ives", "Port_Isaac", "Looe", "Polperro", "Porthleven"
    "East_Sussex", "Brighton", "Battle", "Hastings_(England)", 
    "Rye_(England)", "Seaford", "Ashdown_Forest"
] 

wikivoyage_root_url = "https://en.wikivoyage.org/wiki"

In [27]:
uk_destination_urls = [f'{wikivoyage_root_url}/{d}' for d in uk_destinations]

In [29]:
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url,     header_template=header_template
                                  ) #C
    docs =  html_loader.load() #D
    
    for doc in docs:
        print(doc.metadata)
        granular_chunks = split_docs_into_granular_chunks(docs)
        uk_granular_collection.add_documents(documents=granular_chunks)

        coarse_chunks = split_docs_into_coarse_chunks(docs)
        uk_coarse_collection.add_documents(documents=coarse_chunks)
#A Create a Chroma DB collection
#B Reset the collection in case it already exists 
#C Loader for one destination
#D Documents of one destination 

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.93it/s]


{'source': 'https://en.wikivoyage.org/wiki/Cornwall', 'title': 'Cornwall – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.47it/s]


{'source': 'https://en.wikivoyage.org/wiki/North_Cornwall', 'title': 'North Cornwall – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.13it/s]


...

{'source': 'https://en.wikivoyage.org/wiki/Rye_(England)', 'title': 'Rye (England) – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.40it/s]


{'source': 'https://en.wikivoyage.org/wiki/Seaford', 'title': 'Seaford – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.15it/s]


{'source': 'https://en.wikivoyage.org/wiki/Ashdown_Forest', 'title': 'Ashdown Forest – Travel guide at Wikivoyage', 'language': 'en'}


#### Searching 

In [30]:
granular_results = uk_granular_collection.similarity_search(
    query="Events or festivals in East Sussex",k=4)
for doc in granular_results:
    print(doc)

page_content='Do 
 [ edit ] 
 
 
 
 
   
 Brighton Theatres .   Brighton is a great place to see a theatre show or a gig. There are many many theatres and venues in and around Brighton.   
 
 
 
 50.81715 -0.12348 1   Sealanes  on Madeira Drive is an open-air lido with heated 50 m pool, opening in spring 2023. 
 
 Football:   
 
 50.8618 -0.0833 2   Brighton & Hove Albion ,   Falmer BN1 9BL   ( off A27 5   mi (8.0   km) northeast of the city by Falmer railway station ),   ☏   +44 1273 668855 .   "The Seagulls" play football in the Premier League, England's top tier, with their home ground at Falmer or Amex Stadium, capacity 30,750. Their women's team play in the Women's Super League, with home games at Broadfield Stadium in  Crawley , shared with Crawley Town. In 2025 Falmer Stadium hosted games in the Women's Rugby Union World Cup.         ( updated Sep 2025 ) 
 
 
 Cricket:   
 
 50.8306 -0.1645 3   Sussex CCC ,   Eaton Rd, Hove BN3 3AN   ( 1 mile west of central Brighton ),   ☏   +4

In [31]:
coarse_results = uk_coarse_collection.similarity_search(
    query="Events or festivals in East Sussex",k=4)
for doc in coarse_results:
    print(doc)

page_content='### Events

[edit]

A market during the Brighton Festival. The Theatre Royal is the red building.
A colourful parade down Queens Road during Pride in 2016.

  * **Brighton Racecourse** has flat-racing April-Oct. It's on Freshfield Rd a mile east of town centre.
  * **Plumpton Racecourse** is National Hunt (jumps races) Nov-March, but it's 10 mi (16 km) north in Lewes.
  * Brighton Festival Fringe: early May – early June, ☏ +44 1273 764900, info@brightonfringe.org. The Fringe runs at the same time as the main festival, and features over 600 events, including comedy, theatre, music, and "open houses" (local artists exhibiting in their own homes) and tours (haunted pubs, Regency Brighton, churches, cemeteries, sewers, etc.)_ (date needs fixing)_
  * Brighton Festival: May, ☏ +44 1273 709709 (tickets), tickets@brightonfestival.org. The Brighton Festival, in May each year, is the second biggest arts festival in Great Britain (coming closely behind Edinburgh). Music of all sort

In [29]:
granular_results = uk_granular_collection.similarity_search(
    query="Beaches in Conrwall",k=4)
for doc in granular_results:
    print(doc)

page_content='Cornwall' metadata={'Header 1': 'Cornwall'}
page_content='North Cornwall' metadata={'Header 1': 'North Cornwall'}
page_content='West Cornwall' metadata={'Header 1': 'West Cornwall'}
page_content='South Cornwall' metadata={'Header 1': 'South Cornwall'}


In [30]:
coarse_results = uk_coarse_collection.similarity_search(
    query="Beaches in Cornwall",k=4)
for doc in coarse_results:
    print(doc)

page_content='**South Cornwall** is in Cornwall. It includes much of the stunning Cornish
coast along the English Channel of the Atlantic Ocean.

## Towns and villages

[edit]

Map of South Cornwall

  * 50.26-5.0511 Truro — Cornwall's main centre hosts the Royal Cornwall Museum
  * 50.3311-4.20212 Cawsand — overlooks Plymouth Sound; Cawsand is within Mount Edgcumbe Country Park
  * 50.15-5.073 Falmouth — famous for its beaches, it is home to the world's third largest natural harbour
  * 50.334-4.6334 Fowey — the Fowey Regatta in mid-August attracts many yachts and sailing boats
  * 50.354-4.4545 Looe — a summer resort place with a monkey sanctuary, and an active fishing village
  * 50.408-4.2126 Saltash — "Gateway to Cornwall", a small town on the Cornwall side of the Tamar crossings
  * 50.338-4.7957 St Austell — largest town in the county and home to the Eden Project, the world's largest greenhouse
  * 50.3314-4.75788 Charlestown — seaside town used as filming location for the TV sh

# Embedding strategy

## Embedding child chunks with ParentDocumentRetriever

In [32]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

### Setting up the Parent Document retriever

In [33]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000) #A
child_splitter = RecursiveCharacterTextSplitter(chunk_size=500) #B

child_chunks_collection = Chroma( #C
    collection_name="uk_child_chunks",
    embedding_function=GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=GOOGLE_API_KEY),
)

child_chunks_collection.reset_collection() #D

doc_store = InMemoryStore() #E

parent_doc_retriever = ParentDocumentRetriever( #F
    vectorstore=child_chunks_collection,
    docstore=doc_store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)
#A Splitter to generate parent coarse chunks from original documents (parsed from web pages)
#B Splitter to generate child granular chunks from parent coarse chunks
#C Vector store collection to host child granular chunks
#D Make sure the collection is empty
#E Document store to host parent coarse chunks
#F Retriever to link parent coarse chunks to child granular chunks

### Ingesting the content into doc and vector store

In [34]:
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url, header_template=header_template) #A
    html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(html_docs) #C

    print(f'Ingesting {destination_url}')
    parent_doc_retriever.add_documents(text_docs, ids=None) #D

#A Loader for destination web page
#B HTML documents of one destination 
#C Transform HTML docs into clean text deocs
#D Ingest coarse chunks into document store and granular chunks into vector store

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.03it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.49it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.52it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Rye_(England)


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.17it/s]


Ingesting https://en.wikivoyage.org/wiki/Seaford


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.22it/s]


Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


In [35]:
list(doc_store.yield_keys())
#A Show the keys of the added coarse chunks

['727e14a0-9330-44f3-b699-b673d110c92a',
 '501ef847-4e1c-46f3-8fee-25d8fb3403d9',
 '54567176-2119-43e1-b1a1-39524e22b638',
 '4cf54df3-363c-4959-89b4-2fe13574a044',
 '6f8aaea5-ad3f-4484-945a-a33ffcb57909',
 '89a6b58f-9305-4c16-b686-1a15a79f94c8',
 'a72610f4-2a81-489d-b482-5a74d2404069',
 '2899c167-2d6e-4213-83f1-29c947f4844b',
 '02ccee26-96f4-46bd-a517-c68f1cc2a488',
 'd17a0da7-91e1-4318-a186-72f9a0b7afd4',
 'a47e4070-3e3d-408f-9459-4a6d381f0692',
 '193ec454-73ff-4bd5-b4a7-2d4387298cf6',
 'c52fcf06-d788-4e3b-867a-b89b3cf4a0f4',
 '5efa0224-fb7e-40a6-ac51-61db0cf95efe',
 '25fc7a39-e51b-4f79-b9d0-6b2ef124c69e',
 '01405818-0045-429f-9d00-de6d3cc43c91',
 '89b08e11-8770-4d60-995b-c1a2172b002c',
 '77064d1f-fc36-43cd-b491-f95206500262',
 '7ea0ae35-5289-44a9-bf1f-b9603254eb89',
 'e9bfa935-cf3d-4587-ab11-025f98c7a1b1',
 '52810e0b-b4d0-47eb-bbf6-97a73905b991',
 '0d24937f-16fe-4fa3-8f6f-00faf6ce13dd',
 '22fb27a1-136b-40a2-a5ad-fbfdcdc48ffd',
 '12541726-582b-4211-85c1-650615a6c344',
 '5e8884b6-3f69-

### Performing a search on granular information 

In [36]:
retrieved_docs = parent_doc_retriever.invoke("Cornwall Ranger")

In [37]:
len(retrieved_docs)

4

In [38]:
retrieved_docs[0]

Document(metadata={'source': 'https://en.wikivoyage.org/wiki/Cornwall', 'title': 'Cornwall – Travel guide at Wikivoyage', 'language': 'en'}, page_content="**Go Cornwall Bus** buses operate between:\n\n  * **10** \\- Plymouth to Saltash, Looe and Polperro\n  * **11** \\- Plymouth to Saltash, Liskeard, Bodmin, Wadebridge and Padstow\n\n**Stagecoach** buses operate between Barnstaple, Holsworthy, Launceston and\nTavistock, across the Cornwall and Devon border (**85**).\n\n## Get around\n\n[edit]\n\n### By bus\n\n[edit]\n\nThanks to Transport for Cornwall, all bus tickets are interchangeable across\nthe different companies (except certain town buses in St Ives and Fowey). The\n**Cornwall All Day ticket** allows unlimited travel for a calendar day. As of\n2025, day passes are £8 for adults and £5 for under-19s and £3 singles,\nregardless of age. Payment is by cash or contactless. Real time information\nand timetables can now be found through Transport for Cornwall (most reliable\nfor real t

### Comparing with direct semantic search on child chunks

In [39]:
child_docs_only =  child_chunks_collection.similarity_search("Cornwall Ranger")

In [42]:
child_docs_only

[Document(id='a4771069-5449-430d-a7bb-4d229f9e7288', metadata={'source': 'https://en.wikivoyage.org/wiki/Cornwall', 'doc_id': '2899c167-2d6e-4213-83f1-29c947f4844b', 'title': 'Cornwall – Travel guide at Wikivoyage', 'language': 'en'}, page_content='The **Cornwall Ranger** ticket allows unlimited train travel in Cornwall and\nPlymouth for a calendar day. As of 2023, this costs £14 for adults and £7 for\nunder-16s.\n\n### By ferry/boat\n\n[edit]'),
 Document(id='dabd0d73-54fe-4802-b062-dea14cfdb3a6', metadata={'doc_id': '9e814967-45dd-4bf4-9c49-b57fc68e171b', 'language': 'en', 'source': 'https://en.wikivoyage.org/wiki/West_Cornwall', 'title': 'West Cornwall – Travel guide at Wikivoyage'}, page_content='The **Cornwall Ranger** ticket allows unlimited train travel in Cornwall and\nPlymouth for a calendar day. As of 2023, this costs £14 for adults and £7 for\nunder-16s.\n\n## See\n\n[edit]\n\n**Minack Theatre** is an outdoor theatre built by hand into the side of cliff\nover looking the oce

In [ ]:
%%sql


In [40]:
len(child_docs_only)

4

In [41]:
child_docs_only[0]

Document(id='a4771069-5449-430d-a7bb-4d229f9e7288', metadata={'source': 'https://en.wikivoyage.org/wiki/Cornwall', 'doc_id': '2899c167-2d6e-4213-83f1-29c947f4844b', 'title': 'Cornwall – Travel guide at Wikivoyage', 'language': 'en'}, page_content='The **Cornwall Ranger** ticket allows unlimited train travel in Cornwall and\nPlymouth for a calendar day. As of 2023, this costs £14 for adults and £7 for\nunder-16s.\n\n### By ferry/boat\n\n[edit]')

In [41]:
# IMPORTANT: as you can see a granular search would have identified the chunk, but it would have lost the usefulcontext about travelling in Cornwall

## Embedding child chunks with MultiVectorRetriever

In [43]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import uuid

### Setting up the Multi vector retriever

In [44]:
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000) #A
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500) #B

child_chunks_collection = Chroma( #C
    collection_name="uk_child_chunks",
        embedding_function=GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=GOOGLE_API_KEY),
)

child_chunks_collection.reset_collection() #D

doc_byte_store = InMemoryByteStore() #E
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever( #F
    vectorstore=child_chunks_collection,
    byte_store=doc_byte_store
)
#A Splitter to generate parent coarse chunks from original documents (parsed from web pages)
#B Splitter to generate child granular chunks from parent coarse chunks
#C Vector store collection to host child granular chunks
#D Make sure the collection is empty
#E Document store to host parent coarse chunks
#F Retriever to link parent coarse chunks to child granular chunks

### Ingesting the content into doc and vector store

In [45]:
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url) #A
    html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(
        html_docs) #C

    coarse_chunks = parent_splitter.split_documents(
        text_docs) #D

    coarse_chunks_ids = [str(uuid.uuid4()) for _ in coarse_chunks]
    all_granular_chunks = []
    for i, coarse_chunk in enumerate(
        coarse_chunks): #E
        
        coarse_chunk_id = coarse_chunks_ids[i]
            
        granular_chunks = child_splitter.split_documents(
            [coarse_chunk]) #F

        for granular_chunk in granular_chunks:
            granular_chunk.metadata[doc_key] = coarse_chunk_id #G

        all_granular_chunks.extend(granular_chunks)

    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(
        all_granular_chunks) #H
    multi_vector_retriever.docstore.mset(
        list(zip(coarse_chunks_ids, coarse_chunks))) #I

#A Loader for one destination
#B Documents of one destination 
#C transform HTML docs into clean text docs
#D Split the destination content into parent coarse chunks
#E Iterate over the parent coarse chunks
#F Create child granular chunks form each parent coarse chunk
#G Link each child granular chunk to its parent coarse chunk
#H Ingest the child granular chunks into the vector store
#I Ingest the parent coarse chunks into the document store

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  4.35it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  4.46it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  4.49it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Rye_(England)


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  4.78it/s]


Ingesting https://en.wikivoyage.org/wiki/Seaford


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  4.85it/s]


Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


### Performing a search on granular information

In [46]:
retrieved_docs = multi_vector_retriever.invoke(
    "Cornwall Ranger")

In [47]:
len(retrieved_docs)

4

In [48]:
retrieved_docs[0]

Document(metadata={'source': 'https://en.wikivoyage.org/wiki/West_Cornwall'}, page_content='Please respect our robot policy https://w.wiki/4wJS when crawling us. Contact\nbot-traffic@wikimedia.org if you need higher volumes. (b74d0e2) (30224bb)')

In [49]:
##IMPORTANT: same as Parent Document retriever, but more control and flexibility on how to link child to parent chunks

### Comparing with direct semantic search on child chunks

In [58]:
child_docs_only =  child_chunks_collection.similarity_search_with_vectors(
    "Cornwall Ranger")

In [59]:
len(child_docs_only)

4

In [60]:
child_docs_only[0]

(Document(id='2c47a9e1-21a7-431d-980d-a847909c600d', metadata={'doc_id': 'a38a2264-00b9-479b-85ba-8ffc9a598a8c', 'source': 'https://en.wikivoyage.org/wiki/West_Cornwall'}, page_content='Please respect our robot policy https://w.wiki/4wJS when crawling us. Contact\nbot-traffic@wikimedia.org if you need higher volumes. (b74d0e2) (30224bb)'),
 array([ 0.00230411,  0.01378052, -0.00257397, ..., -0.00217394,
        -0.01242141, -0.00880953], shape=(3072,)))

In [65]:
## IMPORTANT: Same as before

## Embedding summaries with MultiVectorRetriever

In [53]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
import uuid

### Setting up the Multi vector retriever (similar to when embedding child chunks)

In [54]:
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000) #A

summaries_collection = Chroma( #B
    collection_name="uk_summaries",
    embedding_function=GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=GOOGLE_API_KEY),
)

summaries_collection.reset_collection() #C

doc_byte_store = InMemoryByteStore() #D
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever( #E
    vectorstore=summaries_collection,
    byte_store=doc_byte_store
)
#A Splitter to generate parent coarse chunks from original documents (parsed from web pages)
#B Vector store collection to host child granular chunks
#C Make sure the collection is empty
#D Document store to host parent coarse chunks
#E Retriever to link parent coarse chunks to child granular chunks

### Setting up the summarization chain

In [55]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=GOOGLE_API_KEY)

In [56]:
summarization_chain = (
    {"document": lambda x: x.page_content} #A
    | ChatPromptTemplate.from_template("Summarize the following document:\n\n{document}") #B
    | llm
    | StrOutputParser())

#A Grab the text content from the document
#B Instantiate a prompt asking to generate summary of the provided text
#C Send the LLM the instantiated prompt 
#D Extract the summary text from the response

### Ingesting the coarse chunks and related summaries into doc and vector store

In [57]:
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url, header_template=header_template) #A
    html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(
        html_docs) #C

    coarse_chunks = parent_splitter.split_documents(
        text_docs) #D

    coarse_chunks_ids = [str(uuid.uuid4()) for _ in coarse_chunks]
    all_summaries = []
    for i, coarse_chunk in enumerate(
        coarse_chunks): #E
        
        coarse_chunk_id = coarse_chunks_ids[i]
            
        summary_text =  summarization_chain.invoke(
            coarse_chunk) #F
        summary_doc = Document(page_content=summary_text, 
                               metadata={doc_key: coarse_chunk_id})

        all_summaries.append(summary_doc) #G

    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(
        all_summaries) #H
    multi_vector_retriever.docstore.mset(
        list(zip(coarse_chunks_ids, coarse_chunks))) #I

#A Loader for one destination
#B Documents of one destination 
#C transform HTML docs into clean text docs
#D Split the destination content into coarse chunks
#E Iterate over the coarse chunks
#F Generate a summary for the coarse chunk thorugh the summarization chain
#G Link each summary to its related coarse chunk
#H Ingest the summaries into the vector store
#I Ingest the coarse chunks into the document store

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.00it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.23it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.56it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Rye_(England)


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.48it/s]


Ingesting https://en.wikivoyage.org/wiki/Seaford


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.16it/s]


Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


In [71]:
# COMMENT: the code above is similar to when ingesting child chunks, but it is slower because of the summarization step
# which invokes the LLM.
# The processing can be speeded up by parallelizing the outer for loop on the destination urls.

### Performing a search on granular information

In [72]:
retrieved_docs = multi_vector_retriever.invoke("Cornwall travel")

In [73]:
len(retrieved_docs)

4

In [74]:
retrieved_docs

[Document(metadata={'source': 'https://en.wikivoyage.org/wiki/North_Cornwall', 'title': 'North Cornwall – Travel guide at Wikivoyage', 'language': 'en'}, page_content="### By car\n\n[edit]\n\nCornwall can be accessed by road via the A30 which runs from the end of the M5\nat Exeter, all the way through the heart of Devon and Cornwall down to Land's\nEnd. It is a grade-separated expressway as far as Carland Cross near Truro\n(the expressway is expected to be open as far as Camborne (between Redruth and\nHayle) by March 2024). You can also get to Cornwall via the A38, crossing the\nRiver Tamar at Plymouth via the Tamar Bridge, which levies a toll on eastbound\nvehicles. On summer Saturdays and during bank holiday weekends roads to\nCornwall are usually busy.\n\n### By plane\n\n[edit]\n\n50.440833-4.9952781 Cornwall Airport (**NQY** IATA) in Newquay is the main\nairport for the county, with year-round flights only from Aberdeen, Alicante,\nDublin, London Gatwick, and Manchester. During the

### Comparing with direct semantic search on summaries

In [75]:
summary_docs_only =  summaries_collection.similarity_search(
    "Cornwall Travel")

In [76]:
len(summary_docs_only)

4

In [77]:
summary_docs_only

[Document(id='9553aa2d-456e-4a38-8f95-28f072f13992', metadata={'doc_id': '14554480-abf6-4b35-ae7d-8580a27ecc9a'}, page_content="Cornwall offers a diverse array of attractions spanning natural beauty, legends, gardens, historic sites, arts, and heritage, including both independent sites and National Trust properties.\n\n- Natural and legendary sights: King Arthur's Hall and Brown Willy on Bodmin Moor; Dozmary Pool and tales of the Beast of the Moor.\n- Gardens and nature: The Eden Project’s two glass domes; the Lost Gardens of Heligan near Mevagissey.\n- Castles, archaeology, and coastal culture: Tintagel Castle (Arthurian legends and early medieval finds); Minack Theatre (clifftop outdoor theatre and museum); St Michael's Mount.\n- Arts and museums: Tate St Ives (modern art); National Maritime Museum, Falmouth (small-boat collection and other exhibits).\n- Mining and industrial heritage: Historic tin/copper mine sites such as Geevor Tin Mine, Poldark Mine, King Edward Mine, Crown Hill 

In [78]:
# COMMENT: a direct search on summaries retrieves denser information, but it is missing out on useful details. 
# However, you might consider using the summaries directly if after testing they prove adequate.

## Embedding hypothetical questions with MultiVectorRetriever

In [61]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
import uuid
from typing import List
from pydantic import BaseModel, Field

### Setting up the Multi vector retriever (same as when embedding summaries)

In [62]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000) #A

hypothetical_questions_collection = Chroma( #B
    collection_name="uk_hypothetical_questions",
    embedding_function=GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=GOOGLE_API_KEY),
)

hypothetical_questions_collection.reset_collection() #C

doc_byte_store = InMemoryByteStore() #D
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever( #E
    vectorstore=hypothetical_questions_collection,
    byte_store=doc_byte_store
)
#A Splitter to generate parent coarse chunks from original documents (parsed from web pages)
#B Vector store collection to host child granular chunks
#C Make sure the collection is empty
#D Document store to host parent coarse chunks
#E Retriever to link parent coarse chunks to child granular chunks

### Setting up the chain to generate hypothetical questions

In [63]:
class HypotheticalQuestions(BaseModel):
    """A list of hypotetical questions for given text."""

    questions: List[str] = Field(..., description="List of hypothetical questions for given text")

In [64]:
llm_with_structured_output = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", 
    google_api_key=GOOGLE_API_KEY).with_structured_output(
        HypotheticalQuestions
)

In [65]:
hypothetical_questions_chain = (
    {"document_text": lambda x: x.page_content} #A
    | ChatPromptTemplate.from_template( #B
        "Generate a list of exactly 4 hypothetical questions that the below text could be used to answer:\n\n{document_text}"
    )
    | llm_with_structured_output #C
    | (lambda x: x.questions) #D
)

#A Grab the text content from the document
#B Instantiate a prompt asking to generate 4 hypothetical questions on the provided text
#C Invoke the LLM configured to return an object containing the questions as a typed list of strings
#D Grab the list of questions from the response

### Ingesting the coarse chunks and related hypothetical questions into doc and vector store

In [67]:
for destination_url in uk_destination_urls[:3]:
    html_loader = AsyncHtmlLoader(destination_url, header_template=header_template) #A
    html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(
        html_docs) #C

    coarse_chunks = parent_splitter.split_documents(
        text_docs) #D

    coarse_chunks_ids = [str(uuid.uuid4()) for _ in coarse_chunks]
    all_hypothetical_questions = []
    for i, coarse_chunk in enumerate(
        coarse_chunks): #E
        
        coarse_chunk_id = coarse_chunks_ids[i]
            
        hypothetical_questions = hypothetical_questions_chain.invoke(
            coarse_chunk) #F
        hypothetical_questions_docs = [Document(
            page_content=question, metadata={doc_key: coarse_chunk_id})
                    for question 
                    in hypothetical_questions] #G

        all_hypothetical_questions.extend(hypothetical_questions_docs)

    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(
        all_hypothetical_questions) #H
    multi_vector_retriever.docstore.mset(
        list(zip(coarse_chunks_ids, coarse_chunks))) #I

#A Loader for one destination
#B Documents of one destination 
#C transform HTML docs into clean text docs
#D Split the destination content into coarse chunks
#E Iterate over the coarse chunks
#F Generate a list of hypothetical questions for the coarse chunk thorugh the question generation chain
#G Link each hypothetical question to its related coarse chunk
#H Ingest the hypothetical questions into the vector store
#I Ingest the coarse chunks into the document store

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.93it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.27it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.61it/s]


Ingesting https://en.wikivoyage.org/wiki/South_Cornwall


### Performing a search on granular information

In [68]:
retrieved_docs = multi_vector_retriever.invoke(
    "How can you go to Brighton from London?")

In [69]:
len(retrieved_docs)

4

In [70]:
retrieved_docs

[Document(metadata={'source': 'https://en.wikivoyage.org/wiki/Cornwall', 'title': 'Cornwall – Travel guide at Wikivoyage', 'language': 'en'}, page_content="**Land's End Airport (****LEQ**  IATA**)** is between Land's End and St Just,\naround 6 mi (9.7 km) by road from Penzance. It is a small airport with flights\narriving from the Isles of Scilly.\n\nIn next-door Devon, a small number of flights operate into **Exeter (****EXT**\nIATA**)** from UK and European destinations, including from resorts along the\nMediterranean and in the Canary Islands. Stagecoach bus 4A links Exeter\nAirport with Exeter St Davids railway station, for trains to Cornwall.\n\nFurther afield:\n\n  * **Bristol Airport (**BRS**  IATA)** is a large international airport, with arrivals from a large number of destinations throughout Europe. The Bristol Flyer A1 bus route connects the terminal building with Bristol Temple Meads railway station, for trains to Cornwall.\n  * **London Heathrow Airport (**LHR**  IATA)** i

### Inspecting possible questions matching our question through semantic search

In [71]:
hypothetical_question_docs_only = hypothetical_questions_collection.similarity_search(
    "How can you go to Brighton from London?")

In [72]:
len(hypothetical_question_docs_only)

4

In [73]:
hypothetical_question_docs_only

[Document(id='7068ce39-f4f8-4d3d-a117-bc203ef4454b', metadata={'doc_id': 'df6c3fd9-b3f0-4e6b-8019-1a41998e13ed'}, page_content='What are the available train routes and services for traveling from London to various destinations within Cornwall, and what amenities do they offer?'),
 Document(id='4f8cfa46-2f32-4782-ab7f-b37229c12293', metadata={'doc_id': '1eb5c1f4-14d3-45e2-a8e4-3cc2a228558b'}, page_content='What are the different train services that can take me to Penzance from London, and what is the approximate driving distance from London to Penzance?'),
 Document(id='339f0d0a-0ed0-4636-a799-ec32361e9264', metadata={'doc_id': 'ef14e010-1a6c-4ff8-9234-2cda61f6a2d2'}, page_content='How can I get to Cornwall by various modes of transport?'),
 Document(id='58c82cd6-e89e-4c07-989d-ae380866703e', metadata={'doc_id': 'b74d27b8-155a-4fb4-9a8c-edf03285a28a'}, page_content='How can one travel to South Cornwall by train from major cities like London, and which branch lines serve specific towns?'

# Granular chunk expansion with MultiVectorRetriever

In [74]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import uuid

### Setting up the Multi vector retriever

In [77]:
granular_chunk_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500) #A

granular_chunks_collection = Chroma( #B
    collection_name="uk_granular_chunks",
    embedding_function=GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=GOOGLE_API_KEY),
)

granular_chunks_collection.reset_collection() #C

expanded_chunk_store = InMemoryByteStore() #D
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever( #E
    vectorstore=granular_chunks_collection,
    byte_store=expanded_chunk_store
)
#A Splitter to generate granular chunks from original documents (parsed from web pages)
#B Vector store collection to host child granular chunks
#C Make sure the collection is empty
#D Document store to host expanded chunks
#E Retriever to link parent coarse chunks to child granular chunks

### Ingesting granular and expanded chunks into doc and vector store

In [82]:
uk_destination_urls

['https://en.wikivoyage.org/wiki/Cornwall',
 'https://en.wikivoyage.org/wiki/North_Cornwall',
 'https://en.wikivoyage.org/wiki/South_Cornwall',
 'https://en.wikivoyage.org/wiki/West_Cornwall',
 'https://en.wikivoyage.org/wiki/Tintagel',
 'https://en.wikivoyage.org/wiki/Bodmin',
 'https://en.wikivoyage.org/wiki/Wadebridge',
 'https://en.wikivoyage.org/wiki/Penzance',
 'https://en.wikivoyage.org/wiki/Newquay',
 'https://en.wikivoyage.org/wiki/St_Ives',
 'https://en.wikivoyage.org/wiki/Port_Isaac',
 'https://en.wikivoyage.org/wiki/Looe',
 'https://en.wikivoyage.org/wiki/Polperro',
 'https://en.wikivoyage.org/wiki/PorthlevenEast_Sussex',
 'https://en.wikivoyage.org/wiki/Brighton',
 'https://en.wikivoyage.org/wiki/Battle',
 'https://en.wikivoyage.org/wiki/Hastings_(England)',
 'https://en.wikivoyage.org/wiki/Rye_(England)',
 'https://en.wikivoyage.org/wiki/Seaford',
 'https://en.wikivoyage.org/wiki/Ashdown_Forest']

In [83]:
for destination_url in uk_destination_urls[:4]:
    html_loader = AsyncHtmlLoader(destination_url,header_template=header_template) #A
    html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(
        html_docs) #C

    granular_chunks = granular_chunk_splitter.split_documents(
        text_docs) #D

    expanded_chunk_store_items = []
    for i, granular_chunk in enumerate(
        granular_chunks): #E

        this_chunk_num = i #F
        previous_chunk_num = i-1 #F
        next_chunk_num = i+1 #F
        
        if i==0: #F
            previous_chunk_num = None
        elif i==(len(granular_chunks)-1): #F
            next_chunk_num = None

        expanded_chunk_text = "" #G
        if previous_chunk_num: #G
            expanded_chunk_text += granular_chunks[
                previous_chunk_num].page_content
            expanded_chunk_text += "\n"

        expanded_chunk_text += granular_chunks[
            this_chunk_num].page_content #G
        expanded_chunk_text += "\n"

        if next_chunk_num: #G
            expanded_chunk_text += granular_chunks[
                next_chunk_num].page_content
            expanded_chunk_text += "\n"

        expanded_chunk_id = str(uuid.uuid4()) #H
        expanded_chunk_doc = Document(
            page_content=expanded_chunk_text) #I

        expanded_chunk_store_item = (expanded_chunk_id, 
                                     expanded_chunk_doc)
        expanded_chunk_store_items.append(
            expanded_chunk_store_item)

        granular_chunk.metadata[
            doc_key] = expanded_chunk_id #J
            
    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(
        granular_chunks) #K
    multi_vector_retriever.docstore.mset(
        expanded_chunk_store_items) #L

#A Loader for one destination
#B Documents of one destination 
#C transform HTML docs into clean text docs
#D Split the destination content into granular chunks
#E Iterate over the granular chunks
#F determine the index of the current chunk and its previous and next chunks
#G Assemble the text of the expanded chunk by including the previous and next chunk
#H Generate the ID of the expanded chunk
#I Create the expanded chunk document
#J Link each granular chunk to its related expanded chunk
#K Ingest the granular chunks into the vector store
#L Ingest the expanded chunks into the document store

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.79it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.39it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.46it/s]


Ingesting https://en.wikivoyage.org/wiki/South_Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.46it/s]


Ingesting https://en.wikivoyage.org/wiki/West_Cornwall


### Performing a search on granular information

In [84]:
retrieved_docs = multi_vector_retriever.invoke("Cornwall Ranger")

In [85]:
len(retrieved_docs)

4

In [86]:
retrieved_docs[0]

Document(metadata={}, page_content="Buses only serve designated stops when in towns; otherwise, you can flag them\ndown anywhere that's safe for them to stop.\n\n### By train\n\n[edit]\n\n**CrossCountry Trains** and **Great Western Railway** operate regular train\nservices between the main centres of population, the latter company also\nserving a number of other towns on branch lines. For train times and fares\nvisit National Rail Enquiries.\nThe **Cornwall Ranger** ticket allows unlimited train travel in Cornwall and\nPlymouth for a calendar day. As of 2023, this costs £14 for adults and £7 for\nunder-16s.\n\n### By ferry/boat\n\n[edit]\n### By ferry/boat\n\n[edit]\n\nIn certain areas of Cornwall, ferries exist. They can be considered largely\nseparate from other public transport as they are run by private companies.\nFerries in the Fal River can be travelled on using a 'Mussel Card' visitor's\nticket allowing for unlimited travel across the ferries at £27.60 for an adult\nand slightl

### Comparing with direct semantic search on granular chunks

In [97]:
child_docs_only =  child_chunks_collection.similarity_search("Cornwall Ranger")

In [98]:
len(child_docs_only)

4

In [99]:
child_docs_only[0]

Document(id='a18b9a25-f88f-433b-8819-2d80d8a39fcd', metadata={'language': 'en', 'doc_id': 'f69a9d0a-6153-4c74-9f19-83942aeb3876', 'title': 'South Cornwall – Travel guide at Wikivoyage', 'source': 'https://en.wikivoyage.org/wiki/South_Cornwall'}, page_content='The **Cornwall Ranger** ticket allows unlimited train travel in Cornwall and\nPlymouth for a calendar day. As of 2023, this costs £14 for adults and £7 for\nunder-16s.\n\n## See\n\n[edit]\n\nThe **Eden Project** , near St Austell, a fabulous collection of flora from\nall over the planet housed in two space age transparent domes, and a massive\nzip line.')

In [100]:
# COMMENT: the expanded chunk has more useful context